# Lab 4: 量子優位性に向けて

# 第4章: 変分問題 — サンプルベース量子対角化

*ボーナス部分を除いたこのノートブックの QPU 時間の見積もり: Nighthawk r1 で 10 秒、または Heron r2 で 2 秒。（注: これは見積もりにすぎません。実際のランタイムは変わる場合があります。）*

*QPU 使用量の見積もりはバックエンドの実行時間のみを反映しています。キュー待ち、キャリブレーション、ランタイムセッションの遅延により、QPU 時間が数秒だけのワークロードでも、合計実行時間が数分に延びることがあります。*

### 目次

- 演習1: LUCJ 回路を Nighthawk デバイスにマップする
- 演習2: 得られたビット列を調べる
- 演習3: ビット列を回復する
- 演習4: SQD を古典的な参照部分空間で補強する
- 採点演習: 自分の最良の部分空間を見つけよう！

Lab 4a では、量子優位性が何を意味するか、そして [Quantum Advantage Tracker](https://quantum-advantage-tracker.github.io/) が実証を3つのカテゴリー（古典的に検証可能な問題、変分問題、オブザーバブル推定）にどう分類するかを探りました。$ \mathrm{Fe}_4\mathrm{S}_4$ 分子の計算（変分）や Loschmidt Echo の計算（オブザーバブル推定）のような例を見ました。

ここでは2つ目のカテゴリー、**変分問題** を見ていきます。これらは化学と材料科学の問題で、高度なシミュレーションのために基底状態を得ることに焦点を当てます。材料は直接的に量子力学に支配されており、このことは当然ながら量子計算を適用するのに理想的な領域にしています。

このラボでは、**サンプルベース量子対角化（SQD）** を使い、実機の量子コンピューター上で窒素分子（N₂）の **基底状態エネルギー** を推定します。

SQD はハイブリッドな量子・古典手法です。量子コンピューターがパラメーター化された回路を実行して、どの電子配置が最も重要かを *提案* し、一方で古典コンピューターが線形代数を行います。すなわち、その小さくて賢く選ばれた配置の集合の中で分子ハミルトニアンを対角化します。

この章の終わりまでに、化学の回路を2つの異なる IBM デバイストポロジーにマップし、ノイズの多いハードウェアのビット列をきれいにし、実機の量子ハードウェアを使って古典的な総当たり法を打ち負かすことになります。

### 必要なライブラリーをインストールする

In [ ]:
%pip install qiskit-addon-sqd
%pip install ffsim

### インポート

In [ ]:
# 必要なパッケージをインポートする
import warnings
warnings.filterwarnings("ignore")

import copy
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
from typing import cast
from itertools import combinations
from functools import partial

import pyscf
import pyscf.cc
import pyscf.mcscf
import ffsim

from qiskit.circuit import QuantumCircuit, QuantumRegister, CircuitInstruction, Barrier
from qiskit.transpiler import generate_preset_pass_manager
from qiskit.visualization import plot_error_map
from qiskit_ibm_runtime import QiskitRuntimeService, Sampler
from qiskit_ibm_runtime.fake_provider import FakeMiami

from qiskit_addon_sqd.fermion import SCIResult, solve_sci_batch
from qiskit_addon_sqd.subsampling import subsample
from qiskit_addon_sqd.counts import bit_array_to_arrays, bitstring_matrix_to_integers

In [ ]:
from qc_grader.challenges.qgss_2026 import (
    check_progress,
    grade_lab4c_ex1a,
    grade_lab4c_ex1b,
    grade_lab4c_ex2a,
    grade_lab4c_ex2b,
    grade_lab4c_ex3a,
    grade_lab4c_ex3b,
    grade_lab4c_ex4,
    grade_lab4c_exbonus,
)

ラボを通して、`check_progress` 関数を使って、演習をいくつ完了したかを確認できます。

In [ ]:
check_progress()

## 背景: 5分でわかる分子量子化学

このラボを完了するのに化学の背景知識は **必要ありません**。この節では、必要最小限の語彙だけを与えます。

### 分子、軌道、電子

分子を小さなコンサートホールと考えてください。
**分子軌道** はそのホールの座席です。
**電子** は、1つの厳格な規則に従って着席しなければならない客です。すなわち、各座席（軌道）は *あるスピンの客を高々1人* しか収容できません — これがパウリの排他原理です。

各空間軌道は2つの電子を収容できます。1つは **スピン α（↑）**、もう1つは **スピン β（↓）** です。
私たちは常に2つのスピンの種類を別々に追跡します。これがまさに、後で測定するビット列が α の半分と β の半分に分かれる理由です。

### 基底状態と、それがなぜ重要か

**基底状態** は、全エネルギーが最も低い着席の配置です。
それは化学者に、分子が安定かどうか、その結合がどれだけ強いか、そして反応するかどうかを教えます。
基底状態エネルギーを正確に計算することは、量子コンピューターの近未来の最も魅力的な応用の1つです。というのも、可能な着席の数が *指数的* に増えるからです。$n$ 個の軌道と $N_e$ 個の電子については、スピン種ごとに $\binom{n}{N_e}$ 通りの配置があり、これはすぐに、どんな古典コンピューターの手にも負えなくなります。
このラボの N₂ では、26 個の活性軌道とスピンあたり 5 個の電子なので、すでにスピンセクターあたり $\binom{26}{5} = 65{,}780$ 通りの配置があり、完全なヒルベルト空間の次元は両方のスピンセクターの積で、40 億状態を超えます。

基底状態を見つけることは、形式的には **固有値問題** です。

$$\hat{H} |\psi_0\rangle = E_0 |\psi_0\rangle$$

ここで $|\psi_0\rangle$ は基底状態の波動関数、$E_0$ は最も低いエネルギー固有値です。
同等に、**変分原理** はこれを最小化問題として枠組み化します。

$$E_0 = \min_{\|\psi\|=1} \langle \psi | \hat{H} | \psi \rangle$$

あなたが構築するどんな試行状態 $|\psi\rangle$ も、$E_0$ の **上界** を与えます: $\langle\psi|\hat{H}|\psi\rangle \geq E_0$。
この不等式が、SQD を含むあらゆる変分量子アルゴリズムの土台になっています。

第二量子化では、ハミルトニアンは1体項と2体項に分かれます。

$$\hat{H} = \sum_{pq} h_{pq}\,\hat{a}^\dagger_p \hat{a}_q + \frac{1}{2}\sum_{pqrs} g_{pqrs}\,\hat{a}^\dagger_p \hat{a}^\dagger_q \hat{a}_s \hat{a}_r + E_\text{nuc}$$

- $h_{pq}$: **1電子積分** — 運動エネルギーと電子・核間の引力（コード中の `hcore`）。
- $g_{pqrs}$: **2電子反発積分** — 電子・電子間のクーロン反発（コード中の `eri`）。
- $E_\text{nuc}$: 一定の核間反発エネルギー。

SQD はこの問題を、$\hat{H}$ を小さな **部分空間**（注意深く選ばれたスレーター行列式の集合）に射影し、その部分空間の中で厳密に対角化することで解きます。
量子コンピューターの仕事は、どの行列式がそこに属するかを提案することです。

### 古典的な参照手法

3つの古典的なベンチマークが、私たちのプロットの枠組みになります。

| 手法 | 何をするか | 精度 |
|---|---|---|
| **ハートリー・フォック（HF）** | 各電子が他のすべてを無視して、独立に最も安い座席を選ぶ。速いが、電子 *相関*（電子が互いを避けようとする傾向）を無視する。 | $E_0$ の上界。粗い見積もり。 |
| **CCSD**（Coupled Cluster Singles & Doubles） | HF から始め、代数的な振幅 $t_1$（シングル）と $t_2$（ダブル）を使って、HF 配置からの1・2電子の「ジャンプ」を加える。相関のほとんどを捉える。 | $E_0$ の効率的で妥当な古典的見積もりを与える。 |
| **古典的な selected-CI**（このラボでの参照部分空間） | 小さな、手で選んだスレーター行列式の集合の中でハミルトニアンを構築して対角化する。高品質な行列式が与えられれば、ほぼ厳密な精度に近づける。このラボの参照部分空間は総当たりの代用品で、選択基準なしに励起ランクで配置を列挙し、その後厳密に対角化する。あなたの SQD の結果がこのベースラインを上回れば、*量子有用性* を実証したことになる。量子サンプラーが、どんな総当たりの列挙でも到達できない配置を見つけたのだ。 | 配置の品質に応じてスケールする。このラボの総当たり版は、典型的には HF より良いが CCSD には及ばない。 |

> **重要な点:** CCSD は量子回路の種として *使う* こともあります。その振幅は電子が軌道間をどうジャンプしたがるかを記述しており、私たちはその化学的な直観を量子アンザッツのパラメーターに直接移植します。

## 1. N₂ 分子とそのハミルトニアンを構築する

### 窒素分子

**分子状窒素（N₂）** を扱います。2.0 Å 離れた2つの窒素原子です。
これは N₂ の平衡結合長（約 1.1 Å）のおよそ2倍で、電子相関の効果がより顕著になる引き伸ばされた配置です。これにより HF と厳密な基底状態の差が大きくなり、量子計算の応用がより魅力的になります。
分子は **`cc-pvdz` 基底関数系** を使って記述します。これは精度と計算コストのバランスをとる、軌道の形の標準的な語彙です。
この基底でハートリー・フォックを実行すると、合計 28 個の空間軌道が得られます。

### 活性空間と凍結軌道

すべての軌道が同じように重要というわけではありません。
最も内側の **コア** 軌道（各窒素原子の 1s 電子）はエネルギー的に深いところにあり、本質的に決して変化しません。分子が何をしようと、それらはそこにいて完全に満足しています。
それらの軌道を量子力学的に計算するのは、貴重な量子ビットの無駄になります。

そこで、それらを **凍結** します。
- `n_frozen = 2` は、エネルギーが最も低い2つのコア軌道を量子的な扱いから外します。
- `active_space = range(n_frozen, mol.nao_nr())` は、化学的に興味深いこと（結合形成、電子相関）が実際に起こる軌道の集合です。

これが **活性空間近似** です。まず全体について完全な HF を解き、その後、活性空間の電子だけを量子コンピューター上で扱います。
凍結の後、残るのは次のとおりです。
- `num_orbitals` 個の活性空間軌道 → これがスピンセクターあたりの量子ビット数に等しくなります。
- スピン α とスピン β の `(num_elec_a, num_elec_b)` 個の活性電子。N₂ は閉殻の一重項なので、`num_elec_a == num_elec_b` です。

下のコードセルはまた、**1電子積分**（`hcore`）、**2電子反発積分**（`eri`）、そして一定の **核間反発エネルギー** を取り出します。これらを合わせたものが、活性空間に制限された $\hat{H}$ の数値表現であり、後で古典的な固有値ソルバーが対角化します。

In [ ]:
# N2 分子を構築する
mol = pyscf.gto.Mole()
mol.build(
    atom=[["N", (0, 0, 0)], ["N", (2.0, 0, 0)]],
    basis="cc-pvdz",
)

# 活性空間を定義する
n_frozen = 2
active_space = range(n_frozen, mol.nao_nr())

# 分子積分を取得する
scf = pyscf.scf.RHF(mol).run()
num_orbitals = len(active_space)
n_electrons = int(sum(scf.mo_occ[active_space]))
num_elec_a = (n_electrons + mol.spin) // 2
num_elec_b = (n_electrons - mol.spin) // 2
cas = pyscf.mcscf.CASCI(scf, num_orbitals, (num_elec_a, num_elec_b))
mo = cas.sort_mo(active_space, base=0)
hcore, nuclear_repulsion_energy = cas.get_h1cas(mo)
eri = pyscf.ao2mo.restore(1, cas.get_h2cas(mo), num_orbitals)

# 分子の情報を出力する
print(f"Number of molecular orbitals: {num_orbitals}")
print(f"Number of electrons (a, b): {(num_elec_a, num_elec_b)}")

### 量子アンザッツの種にするために CCSD を実行する

量子回路を構築する前に、**CCSD** を古典的に実行します。
CCSD を最終的な答えとして *使うわけではありません* — それでは意味がなくなってしまいます。
その **振幅を採掘** しているのです。

- `t1[i, a]` — 1つの電子が占有軌道 $i$ から仮想軌道 $a$ へ「ジャンプ」する振幅。
- `t2[i, j, a, b]` — 電子 *対* が $(i, j)$ から $(a, b)$ へ散乱する振幅。

これらの振幅は、電子がどう相関するかについての CCSD の化学的な直観をエンコードしています。
次の節では、`ffsim` ライブラリーがそれらを読み取って量子回路の回転角にコンパイルし、事実上、古典的な化学の知識を高品質な初期推定として量子デバイスに移植します。

In [ ]:
# アンザッツを初期化するために CCSD の t2 振幅を取得する
ccsd = pyscf.cc.CCSD(scf, frozen=[i for i in range(mol.nao_nr()) if i not in active_space]).run(max_cycle=1000)
t1 = ccsd.t1
t2 = ccsd.t2

## 2. 量子状態の準備: LUCJ アンザッツ

### LUCJ アンザッツとは何か？

状態準備の目標は、N₂ の基底状態を十分よく近似する量子回路を構築し、それをサンプリングすると真の基底状態に近い電子配置が得られるようにすることです。
**LUCJ アンザッツ** — **Local Unitary Cluster Jastrow** — を使います。これはハードウェア効率の良いパラメーター化された回路で、2つの要素を通して電子相関を捉えます。

1. **軌道回転** $e^{i\hat{K}}$: 分子軌道を混ぜる1体ユニタリー（座席を並べ替えると考えてください）。
2. **Jastrow 因子** $e^{i\sum_{pq} J_{pq}\,\hat{n}_p \hat{n}_q}$: 軌道 $p$ と $q$ を同時に占有する電子対にペナルティーや報酬を与える対角的な相互作用（電子が互いの存在に反応します）。

1つの UCJ 層は次の形を持ちます。

$$U_\mu = e^{i\hat{K}_\mu}\; e^{i\sum_{p<q} J^{(\mu)}_{pq}\,\hat{n}_p \hat{n}_q}\; e^{-i\hat{K}_\mu}$$

ここで $\hat{n}_p = a^\dagger_p a_p$ は軌道 $p$ の占有数演算子です。
ここで $a^\dagger_p$ は軌道 $p$ に電子を **生成** し、$a_p$ は1つを **消滅** させます。これは多体量子系のための標準的な **第二量子化** の言葉です。
$\hat{n}_p$ は単に占有状態への射影で、軌道 $p$ が占有されていれば 1、空なら 0 に等しくなります。
完全なアンザッツは、そのような層を `n_reps` 個積み重ねます: $U = U_{n_\text{reps}} \cdots U_1$。
ここでは `n_reps = 1` を使います。

### CCSD 振幅から回路パラメーターへ

`ffsim.UCJOpSpinBalanced.from_t_amplitudes(t1, t2, n_reps, interaction_pairs)` は、CCSD の1電子・2電子励起振幅を読み取り、回転角 $K_\mu$ と Jastrow 結合 $J_{pq}^{(\mu)}$ にコンパイルします。
直観的には、CCSD が「軌道 $i$ と $a$ が強く入れ替わる」と言えば、$e^{i\hat{K}}$ の対応する軌道回転が大きな角度を得ます。

**スピンバランス（Spin-balanced）** とは、α（スピンアップ）と β（スピンダウン）のセクターが同じ軌道回転パラメーターを共有することを意味します。N₂ のような閉殻の一重項（α と β の占有が同じ）に適しており、自由なパラメーターの数を半分にします。

### 相互作用ペア — ハードウェアの制約

Jastrow 因子は、2つの量子ビットが直接相互作用することを要求します。
`interaction_pairs` 引数は、どの軌道のペアが相互作用できるかを `ffsim` に伝えます。

- `alpha_alpha_indices = [(p, p+1) ...]` — 量子ビット鎖に沿った同一スピンの *最近接* 相互作用（チップ上でスピンセクターごとに1本の線）。
- `alpha_beta_indices` — **スピンをまたぐ** 相互作用: 位置 $p$ の α 量子ビットが位置 $q$ の β 量子ビットと結合する。
  これらは **デバイスの配線によって制約されます**: 2つの物理量子ビットがチップ上で隣接している場合にのみ、その結合はハードウェアネイティブです。

これが演習1の重要な点です。異なる IBM デバイスは異なるトポロジーを持つので、許される `alpha_beta_indices` は Heron チップと Nighthawk チップで異なります。

In [ ]:
service = QiskitRuntimeService()
backend_hr = service.backend("ibm_kingston")
plot_error_map(backend_hr)

### ジョルダン・ウィグナー変換: 軌道から量子ビットへ

**ジョルダン・ウィグナー（JW）変換** は、フェルミオンの軌道状態を量子ビットの状態に対応させる辞書です。

$$\text{軌道 } p \text{ が占有} \;\Longleftrightarrow\; \text{量子ビット } p = |1\rangle, \qquad
  \text{軌道 } p \text{ が空} \;\Longleftrightarrow\; \text{量子ビット } p = |0\rangle$$

合計で $2 \times \texttt{num\_orbitals}$ 個の量子ビットを使います。
- 量子ビット $0, \ldots, \texttt{num\_orbitals}-1$ → スピン α の占有
- 量子ビット $\texttt{num\_orbitals}, \ldots, 2\cdot\texttt{num\_orbitals}-1$ → スピン β の占有

これがまさに、ずっと後の演習2a で、長さ `2*num_orbitals` の測定されたビット列が、α の半分と β の半分にきれいに分かれる理由です。

2つの `ffsim` の回路命令がこのエンコードを扱います。
- `PrepareHartreeFockJW` — 初期の量子ビット状態をハートリー・フォックの占有パターンに設定します（最初の `num_elec_a` 個の α 量子ビットと `num_elec_b` 個の β 量子ビットが $|1\rangle$、残りが $|0\rangle$）。
- `UCJOpSpinBalancedJW` — JW 量子ビット基底で UCJ 相関子を適用します。

### PRE_INIT パスとゲート数

`ffsim.qiskit.PRE_INIT` トランスパイラーパスは、メインのトランスパイルパスが走る *前に*、高レベルの `ffsim` ゲートオブジェクトをネイティブなハードウェアゲートに分解します。
`PRE_INIT` あり・なしで `count_ops()` を実行すると、回路が必要とするネイティブな2量子ビットゲートの数が分かります。ゲートが少ないほど、蓄積するノイズが少なくなります。

下のコードは、**Heron（`ibm_kingston`）** デバイスの完全なセットアップを実例として示します。
演習1では、**Nighthawk（`ibm_miami`）** デバイスについてこのマッピングをやり直すことを求めます。

### SQD ワークフロー: 簡単な地図

回路の準備ができたら、この章の残りは4ステップのループに従います。

1. **サンプリング** — LUCJ 回路をハードウェア上で実行し、ビット列を集める。
2. **配置を回復する** — ノイズが誤った電子数へ押しやってしまったビット列を修正する。
3. **部分空間を構築する** — 最も頻繁な（最も確率の高い）電子配置を基底として集める。
4. その部分空間の中で $\hat{H}$ を **対角化する** — エネルギー推定値を得て、得られた軌道占有数をステップ2に戻す。

反復のたびに、部分空間は少しずつ賢くなります。

In [ ]:
# Heron デバイスでは、alpha-beta 相互作用は4つのインデックスごとに起こる
alpha_alpha_indices = [(p, p + 1) for p in range(num_orbitals - 1)]
alpha_beta_indices_hr = [(p, p) for p in range(0, num_orbitals, 4)]
alpha_beta_indices_hr = alpha_beta_indices_hr[:5]  # 5番目のペアで切り詰める

print(alpha_beta_indices_hr)

In [ ]:
n_reps = 1
ucj_op_hr = ffsim.UCJOpSpinBalanced.from_t_amplitudes(
    t1=t1,
    t2=t2,
    n_reps=n_reps,
    interaction_pairs=(alpha_alpha_indices, alpha_beta_indices_hr),
    optimize=True,
    options=dict(maxiter=20),
)
nelec = (num_elec_a, num_elec_b)

# 空の量子回路を作る
qubits = QuantumRegister(2 * num_orbitals, name="q")
circuit_hr = QuantumCircuit(qubits)

# 参照状態としてハートリー・フォック状態を準備し、量子回路に追加する
circuit_hr.append(ffsim.qiskit.PrepareHartreeFockJW(num_orbitals, nelec), qubits)

# 参照状態に UCJ 演算子を適用する
circuit_hr.append(ffsim.qiskit.UCJOpSpinBalancedJW(ucj_op_hr), qubits)

# 実行のために回路を調べる
inner_circuit_hr = ffsim.qiskit.PRE_INIT.run(circuit_hr)
for i in range(2, 0, -1):
    inner_circuit_hr.data.insert(i, CircuitInstruction(Barrier(num_qubits=2*num_orbitals), qubits))
display(inner_circuit_hr.decompose().draw("mpl", fold=-1))

# 最終測定を挿入する
circuit_hr.measure_all()

In [ ]:
# デバイスのトポロジーに従ってスピン a と b のレイアウトを選ぶ
spin_a_layout = [
     21,  36,  41,  42,  43,     56,  63,  64,  65,  77,
     85,  86,  87,  97, 107,    108, 109, 118, 129, 128,
    127, 126, 125, 117, 105,    104
]
spin_b_layout = [
     23,  24,  25,  37,  45,     46,  47,  57,  67,  68,
     69,  78,  89,  90,  91,     98, 111, 112, 113, 114,
    115,  99,  95,  94,  93,     79
]

initial_layout = spin_a_layout + spin_b_layout

pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend_hr, initial_layout=initial_layout
)

# PRE_INIT パスなし
isa_circuit_hr = pass_manager.run(circuit_hr)
print(f"Gate counts (w/o pre-init passes): {isa_circuit_hr.count_ops()}")

# PRE_INIT パスあり
# このパスマネージャーが生成する回路をハードウェア実行に使う
pass_manager.pre_init = ffsim.qiskit.PRE_INIT
isa_circuit_hr = pass_manager.run(circuit_hr)
print(f"Gate counts (w/ pre-init passes): {isa_circuit_hr.count_ops()}")

## 演習1: Nighthawk デバイス用に LUCJ 回路をマップする

では、同じ回路を **Nighthawk（`ibm_miami`）** デバイスに移植しましょう。
下のエラーマップのセルを実行し、接続性の図を Heron のものと比較してください。
α と β の量子ビットのレールがどう違う配線になっているかに注目してください。それによって、どのスピンをまたぐ相互作用ペアがハードウェアネイティブになるかが変わります。

> **`ibm_miami` にアクセスできない？** 代わりに `FakeMiami` を使ってください。下のセルの2行目のコメントを外します。`FakeMiami` は Nighthawk デバイスのトポロジー（同じ量子ビットグラフと基底ゲート）を再現するので、回路のマッピングとレイアウトの演習は同一に動作します。唯一の違いは、実機で実行するのではなくノイズをシミュレートする点です。

In [ ]:
backend_nh = service.backend("ibm_miami")
# # ibm_miami にアクセスできない場合は、このコマンドで FakeMiami を使う
# backend_nh = FakeMiami()

plot_error_map(backend_nh)

<a id="exercise_1a"></a>
<div class="alert alert-block alert-success">

<b>演習1a: Nighthawk デバイス用の α–β 相互作用ペアを選ぶ</b>

**目標:** Nighthawk デバイスのトポロジーに適したスピンをまたぐ相互作用ペアで `alpha_beta_indices_nh` を埋めてください。

**背景:** 2節で述べたように、`alpha_beta_indices` の各 `(p, q)` タプルは「Jastrow 因子が α 軌道 $p$ を β 軌道 $q$ に結合し、その2つの量子ビットはチップ上で物理的に隣接していなければならない」ことを意味します。

- **Heron（`ibm_kingston`）** では、α と β の量子ビット鎖は4つおきの位置でしか接触しないので、`[(p, p) for p in range(0, num_orbitals, 4)]` を5ペアに切り詰めて使いました（ハードウェア的に隣接する α–β の交差は5つしかありません）。
- **Nighthawk（`ibm_miami`）** では、α と β のレールを隣り合わせに並べられるので、α 鎖の $p$ 番目の量子ビットが *すべての* $p$ について β 鎖の $p$ 番目の量子ビットと相互作用できます。インデックスを飛ばす必要はありません。

**ヒント:**
- **（推奨）** すべての量子ビットをペアにするのではなく、24 個の量子ビットペアを作れます（24 で切り詰める）。こうすると回路を Nighthawk トポロジーにマップしやすくなります。
- すべての α と β のペアを隣り合わせに配置できないことに気づくかもしれませんが、この段階では気にしないでください。

</div>

In [ ]:
# Nighthawk デバイスのトポロジーに従って alpha-beta 相互作用ペアを選ぶ
# ---- TODO : タスク1a ----
alpha_beta_indices_nh = []
# ---- TODO : タスク1a ここまで ----

print(alpha_beta_indices_nh)

In [ ]:
grade_lab4c_ex1a(alpha_beta_indices_nh)

In [ ]:
ucj_op_nh = ffsim.UCJOpSpinBalanced.from_t_amplitudes(
    t1=t1,
    t2=t2,
    n_reps=n_reps,
    interaction_pairs=(alpha_alpha_indices, alpha_beta_indices_nh),
    optimize=True,
    options=dict(maxiter=20),
)
nelec = (num_elec_a, num_elec_b)

# 空の量子回路を作る
qubits = QuantumRegister(2 * num_orbitals, name="q")
circuit_nh = QuantumCircuit(qubits)

# 参照状態としてハートリー・フォック状態を準備し、量子回路に追加する
circuit_nh.append(ffsim.qiskit.PrepareHartreeFockJW(num_orbitals, nelec), qubits)

# 参照状態に UCJ 演算子を適用する
circuit_nh.append(ffsim.qiskit.UCJOpSpinBalancedJW(ucj_op_nh), qubits)

# 実行のために回路を調べる
inner_circuit_nh = ffsim.qiskit.PRE_INIT.run(circuit_nh)
for i in range(2, 0, -1):
    inner_circuit_nh.data.insert(i, CircuitInstruction(Barrier(num_qubits=2*num_orbitals), qubits))
display(inner_circuit_nh.decompose().draw("mpl", fold=-1))

# 最終測定を挿入する
circuit_nh.measure_all()

<a id="exercise_1b"></a>
<div class="alert alert-block alert-success">

<b>演習1b: Nighthawk デバイス用の量子ビットレイアウトを定義する</b>

**目標:** `spin_a_layout` と `spin_b_layout` を与えてください。それぞれちょうど `num_orbitals`（= 26）個の物理量子ビットインデックスのリストで、α と β のスピン軌道鎖を Nighthawk のハードウェアに正しくマップするものです。

**背景:** `initial_layout` は、論理量子ビット $p$ を物理量子ビット `initial_layout[p]` に配置するようトランスパイラーに指示します。良いレイアウトは、2つの要件を同時に満たさなければなりません。

1. **各スピンセクター内の接続性:** 同一スピンの最近接相互作用 `alpha_alpha_indices = [(p, p+1) ...]` は、各リスト内の物理量子ビット $p$ と $p+1$ がハードウェア的に隣接している（チップ上で2量子ビットゲートの線でつながっている）ことを要求します。したがって各リストは、デバイスグラフを通る *つながった経路* をたどるべきです。

2. **α–β の隣接:** 演習1a で与えたペアに従い、ペアを互いにできるだけ近くに配置します。必要なら数個のスワップを使ってもまったく問題ありません。

**ヒント:**
- すべての α と β のペアが隣り合わせに座るわけではありません。数個のスワップ操作を適用するのは問題ありません。スワップが並列に動作でき、回路の深さを最小に保てる限りは大丈夫です。
- **（オプション）** より発展的な課題に挑戦したい場合は、前の演習で 26 個すべての量子ビットをペアにして、それらを Nighthawk にマップしてみてください。合格の基準はかなり厳しく、grader の結果はトランスパイラーのシードによって変動することがあります。より良いトランスパイル結果を得るために、いくつかの異なるシードを試してください。
</div>

In [ ]:
# デバイスのトポロジーに従ってスピン a と b のレイアウトを選ぶ
# ---- TODO : タスク1b ----
spin_a_layout = [

]
spin_b_layout = [
    
]
initial_layout = spin_a_layout + spin_b_layout
# ---- TODO : タスク1b ここまで ----

pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend_nh, initial_layout=initial_layout
)

# PRE_INIT パスなし
isa_circuit_nh = pass_manager.run(circuit_nh)
print(f"Gate counts (w/o pre-init passes): {isa_circuit_nh.count_ops()}")

# PRE_INIT パスあり
# このパスマネージャーが生成する回路をハードウェア実行に使う
pass_manager.pre_init = ffsim.qiskit.PRE_INIT
isa_circuit_nh = pass_manager.run(circuit_nh)
print(f"Gate counts (w/ pre-init passes): {isa_circuit_nh.count_ops()}")

In [ ]:
grade_lab4c_ex1b(initial_layout, alpha_beta_indices_nh, seed=None)

## 3. 量子サンプリング

回路を Nighthawk のトポロジーにマップしたので、デバイスに 2,000 ショットを投入します。
各 **ショット** はすべての `2*num_orbitals` 個の量子ビットを測定し、1つの **ビット列** を返します。これは、その瞬間にどの軌道が占有されているかの2進のスナップショットです。
物理的には、各ビット列は *1つの提案された電子配置* であり、基底状態の基底状態（basis state）の候補です。

2,000 個すべてのビット列と、各相異なる配置がどれだけ頻繁に現れるかを合わせたものが、SQD が処理する生の材料になります。

> **ランタイムについての注:** 実機のジョブの実行には、`ibm_miami` で 10 秒、`ibm_kingston` で 2 秒かかります。
> このノートブックを自己完結に保つため、下のセルはジョブを投入して `job_id` を保存します。
> *次の* セルは、その保存された ID から事前計算済みの結果を取得します。再投入する必要はありません。

In [ ]:
# # オプション1: Nighthawk デバイスにアクセスできる場合は、正方格子の回路を実行してみる。
# sampler = Sampler(mode=backend_nh)
# job = sampler.run([isa_circuit_nh], shots=2_000)

# # オプション2: Nighthawk デバイスにアクセスできない場合は、Heron 用に合わせた回路を実行できる。
# sampler = Sampler(mode=backend_hr)
# job = sampler.run([isa_circuit_hr], shots=2_000)

job_id = job.job_id()
print(f"Submitted job {job_id}")

下のセルは、完了したジョブを取得し、生の測定結果を2つの配列に変換します。

- `raw_bitstrings` — 形状 `(n_unique, 2*num_orbitals)` の整数配列で、各行が相異なるビット列です（各要素は 0 か 1）。
- `raw_probs` — 長さ `n_unique` の浮動小数点配列で、各ビット列の経験的な確率を与えます。`raw_probs[k]` は `raw_bitstrings[k]` を生んだショットの割合です。

In [ ]:
# ジョブ ID を使ってジョブを取得する
job_id = "d8oks6e8aqlc73egmkl0"
job = service.job(job_id)
primitive_result = job.result()
pub_result = primitive_result[0]
bit_array = pub_result.data.meas

# BitArray をビット列と確率の配列に変換する
raw_bitstrings, raw_probs = bit_array_to_arrays(bit_array)

## 演習2: 得られたビット列を調べる

サンプリングが終わったら、手元にあるものを理解する番です。
下の2つのコードセルで、処理を始める前に `raw_bitstrings` の形状と内容を調べられます。
まずそれらを実行し、その後2つの演習を完了してください。

In [ ]:
print(raw_bitstrings.shape)

In [ ]:
print(raw_bitstrings[0])

<a id="exercise_2a"></a>
<div class="alert alert-block alert-success">

<b>演習2a: 生のビット列をスピンセクターに整形する</b>

**目標:** 実験結果から **スピン α** と **スピン β** のビット列を分割するために `reshape_bitstring` を実装してください。

**背景:** ジョルダン・ウィグナー変換は、スピン α の量子ビットを各ビット列の *最初の* ブロックに、スピン β の量子ビットを *2番目の* ブロックに配置します。
α ブロックの位置 $p$ にある `1` は「α 軌道 $p$ が占有されている」ことを意味し、`0` は空を意味します。
2つのスピンセクターを分けることで、次のことができます。
- スピンごとの電子数を独立に確認する（演習2b）。
- 配置回復の際にスピンごとの補正を適用する（演習3）。

**ヒント:**
- `raw_bitstrings` の形状は **(2000, 52)** です。形状 **(2000, 2, 26)** の配列が得られるはずです。
- **ビット順序の注意:** Qiskit は測定結果を *リトルエンディアン* で返します（最後に測定された量子ビット = 文字列の最初のビット）。整形の後、`reshaped_bitstrings[0, :]` を出力して、どちらのブロックが α かを確認し、そのブロックが α のビット列と一致するかを調べてください。インデックス 0 が常に α になるように、スピンブロックの順序を反転する必要があるかもしれません。
- 上で出力した生の `raw_bitstrings[0]` と突き合わせて確認してください。

</div>

In [ ]:
def reshape_bitstring(
    bitstrings: np.ndarray,
    num_orbitals: int
) -> np.ndarray:

    # ---- TODO : タスク2a ----

    # ---- TODO : タスク2a ここまで ----

    return reshaped_bitstrings

reshaped_bitstrings = reshape_bitstring(raw_bitstrings, num_orbitals)
print(reshaped_bitstrings[0, :])

In [ ]:
grade_lab4c_ex2a(reshape_bitstring, raw_bitstrings)

<a id="exercise_2b"></a>
<div class="alert alert-block alert-success">

<b>演習2b: サンプルごとの電子数を数える（ハミング重み）</b>

**目標:** 整形されたビット列の配列を受け取り、各サンプルの各スピンセクターについて占有された軌道の数を与える整数配列を返す `hamming_weight` を実装してください。

**背景:** 2進文字列の **ハミング重み** は、単にその中の 1 の数です。
私たちの文脈では、ハミング重み = そのサンプルにおけるそのスピンの電子数です。

*有効な* サンプルは、ちょうど 5 個のスピン α 電子 **かつ** 5 個のスピン β 電子を持たなければなりません（粒子数保存）。
ノイズのない世界では、すべてのショットが有効です。実機のハードウェアでは、ビット反転エラーやその他のノイズ機構が、一部のビット列を正しい粒子数セクターの外へ押しやります。それらのサンプルは物理的に無意味で、そのまま使うと対角化を汚染します。

ハミング重みを数えることが、これらの悪いサンプルを検出する最初のステップです。

**ヒント:** 最後の軸（軌道の軸）について和をとってください。

</div>

In [ ]:
def hamming_weight(bitstrings: np.ndarray) -> np.ndarray:

    # ---- TODO : タスク2b ----

    # ---- TODO : タスク2b ここまで ----

    return weight

print(hamming_weight(reshaped_bitstrings)[0])

In [ ]:
grade_lab4c_ex2b(hamming_weight)

下のセルは、どれだけのサンプルがすでに正しい電子数を持っているか、すなわち、いかなる修復の前に何ショットが「有効」かを確認します。この数がゼロでも驚かないでください。実機のハードウェアのビット反転エラーは、簡単にサンプルを正しい粒子数セクターの外へ押しやります。

In [ ]:
print(int(2e3*np.sum(raw_probs[np.all(hamming_weight(reshaped_bitstrings) == nelec)])))

## 演習3: ビット列を回復する

### なぜ配置回復が必要なのか

真の分子の基底状態は、完全に **固定された粒子数のセクター** の中にあります。すなわち、ちょうど `(num_elec_a, num_elec_b)` 個の電子を持つ配置です。
量子力学は粒子数を保存します。物理的な分子が自発的に電子を得たり失ったりすることはできません。

しかし、実機の量子デバイスは **ビット反転エラー**（およびその他のノイズ）を持ち込み、いくつかの `0` を `1` に、あるいはその逆に変えます。
1回のビット反転がビット列を物理的なセクターの *外* へ動かし、突然、そのサンプルは誤った電子数を報告します。
そのサンプルは物理的に無意味で、対角化器に直接与えると、非物理的な配置で部分空間を汚染します。

**配置回復** は、正しい電子数を復元するのに必要な最小限のビットを反転することで、壊れた各サンプルを修復します。*どの* ビットを反転するかは、各軌道の平均占有数についての現在の最良推定に基づいて選びます。

### ハートリー・フォックから占有数を初期化する

回復アルゴリズムは、「各軌道は平均してどれだけ占有されているか？」の事前推定を必要とします。
これを **ハートリー・フォックの占有数** でブートストラップします。すなわち、インデックスが最も高い `num_elec_a` 個の軌道が完全に占有され（占有数 = 1）、残りは空（占有数 = 0）です。
SQD ループが進むにつれて、これは対角化された量子状態から測定された実際の軌道占有数に置き換えられます。したがって、回復は反復のたびに改善します。

In [ ]:
initial_occupancy = np.zeros((2, num_orbitals), dtype=float)
initial_occupancy[:, -num_elec_a:] = 1.
print(initial_occupancy)

<a id="exercise_3a"></a>
<div class="alert alert-block alert-success">

<b>演習3a: ビット反転確率の関数を定義する</b>

**目標:** $[0, 1]$ の軌道占有数の配列が与えられたときに、各軌道を空（0）から占有（1）へ反転する確率の重みを返す `weight_flip_0_to_1(occupancy)` を実装してください。

**背景:** サンプルが電子を *欠いている*（1 が少なすぎる）とき、いくつかの空の軌道を占有に反転しなければなりません。
*ふだん* 占有されている軌道を優先的に埋めたいと思います。真の基底状態はおそらくそれらの軌道が埋まっているからです。
逆に、ふだん空である軌道はほとんど埋めるべきではありません。

次の関数

$$W_{0\to1}(\text{occ}) = e^{\text{occ}} - 1, \qquad 0 \le \text{occ} \le 1$$

は、まさにこの振る舞いを捉えます。
- $W_{0\to1}(0) = 0$ — 一度も占有されない軌道は決して埋めない。
- $W_{0\to1}(1) = e - 1 \approx 1.718$ — 常に占有される軌道を埋めることを強く好む。
- その間で単調増加。

対になる関数 `weight_flip_1_to_0(occ) = weight_flip_0_to_1(1 - occ)` はすでに提供されています。電子を取り除くときは、ふだん空である軌道（0 である確率が高い）を優先します。

</div>

In [ ]:
def weight_flip_0_to_1(occupancy: np.ndarray) -> np.ndarray:

    # ---- TODO : タスク3a ----

    # ---- TODO : タスク3a ここまで ----

    return prob

def weight_flip_1_to_0(occupancy):
    return weight_flip_0_to_1(1-occupancy)

In [ ]:
grade_lab4c_ex3a(weight_flip_0_to_1)

<a id="exercise_3b"></a>
<div class="alert alert-block alert-success">

<b>演習3b: ビット列の配置を正しい電子数に回復する</b>

**目標:** スピンごとのループの内側の3分岐の補正ロジックを埋めて、`recover_configurations` 関数を完成させてください。

**背景 — アルゴリズムの手順:**

外側のループは、すべてのビット列サンプルを反復します。各サンプルと各スピンセクター `i`（0 = α、1 = β）について:

1. `weight[i]` = ハミング重み = セクター `i` の電子数を計算する（演習2b で書いたもの）。
2. `weight[i]` を `num_elec[i]` と比較する:

   - **電子が少なすぎる**（`weight[i] < num_elec[i]`）: `num_elec[i] - weight[i]` 個の空のビットを 1 に反転しなければならない。
     - 空の軌道を見つける
     - それらの候補の反転の重みを取得する
     - 重みを正規化して確率分布を作る
     - どれを反転するかをランダムに選ぶ
     - 選んだビットを `True` に設定する

   - **電子が多すぎる**（`weight[i] > num_elec[i]`）: `weight[i] - num_elec[i]` 個の占有されたビットを 0 に反転しなければならない。
     - 上の手順を鏡写しにするが、反転するのは占有されたビットを選ぶようにする。

   - **ちょうど正しい**（`weight[i] == num_elec[i]`）: そのスピンセクターは変更しない。

**ヒント:** 各分岐は4行必要です。`np.where` で候補を見つけ、確率を計算して正規化し、`rand_seed.choice` を呼んでどのビットを反転するかを選び、そしてビットを反転します。

**注:** 出力の形式: 補正された各ビット列は文字列に変換され、`corrected_dict`（補正後に同一になったビット列をまとめる `defaultdict`）に蓄積されます。

</div>

In [ ]:
def recover_configurations(
    bitstrings: np.ndarray,
    probs: np.ndarray,
    occupancy: np.ndarray,
    num_orbitals: int,
    num_elec: tuple[int, int],
    rand_seed: np.random.Generator,
) -> tuple[np.ndarray, np.ndarray]:
    corrected_dict: defaultdict[str, float] = defaultdict(float)
    for bitstring, freq, weight in zip(bitstrings, probs, hamming_weight(bitstrings)):
        bs_corrected = bitstring.copy()
        for i in range(2):
            # ---- TODO : タスク3b ----



            # ---- TODO : タスク3b ここまで ----
        bs_str = "".join("1" if bit else "0" for bit in bs_corrected.flatten())
        corrected_dict[bs_str] += freq

    bs_mat_out = np.array([[bit == "1" for bit in bs] for bs in corrected_dict]).reshape(-1, 2, num_orbitals)
    freqs_out = np.array([f for f in corrected_dict.values()])
    freqs_out = np.abs(freqs_out) / np.sum(np.abs(freqs_out))

    return bs_mat_out, freqs_out

recovered_bitstrings, recovered_probs = recover_configurations(
    reshaped_bitstrings, raw_probs, initial_occupancy, num_orbitals, nelec, np.random.default_rng())

下のセルは、配置回復の **後** に何個のサンプルが有効か（正しい電子数か）を示します。演習2b の回復前の数と比較してください。

In [ ]:
print(np.rint(2e3*np.sum(recovered_probs[np.all(hamming_weight(recovered_bitstrings) == nelec, axis=1)])).astype(int))

In [ ]:
grade_lab4c_ex3b(recover_configurations)

### 回復されたビット列から対角化の部分空間へ

すべてのサンプルが正しい粒子数セクターに入ったので、対角化のためにそれらを準備します。

1. **`subsample`** は、回復された配置を、それぞれ `samples_per_batch` 個のサンプルからなる `num_batches` 個の独立したバッチに分割します。複数の独立したバッチで固有値ソルバーを実行することで、分散の推定（後の箱ひげ図が広がりを示します）が得られ、運の悪い配置の集合に行き詰まる可能性が減ります。

2. **`bitstring_matrix_to_integers`** は、各2進のビット列を単一の Python 整数にパックします。これは、selected-CI 固有値ソルバー `solve_sci_batch` が期待する標準的な「行列式ラベル」形式です。
   これらの整数ラベルは **CI 文字列**（Configuration Interaction 文字列）と呼ばれます。各整数はどの軌道が占有されているかをエンコードし、1つの **スレーター行列式** を一意に識別します。スレーター行列式とは、パウリの排他原理を満たし1つの有効な電子配置を表す、適切に反対称化された1電子軌道状態の積です。

3. **`build_subspace`** は、CI 文字列を受け取り、各バッチ内で重複を除き、周辺確率でソートし（最もサンプルされたものが先）、必要ならユーザー指定の `include` 集合を先頭に付け、スピンごとに `max_dim` 個の配置に切り詰めます。結果が **部分空間の基底** です。すなわち、ハミルトニアンが射影されるスレーター行列式の集合です。

In [ ]:
subsamples = np.array(subsample(recovered_bitstrings.reshape(-1, 2*num_orbitals), recovered_probs, samples_per_batch=20, num_batches=5))
subsamples = subsamples.reshape(*subsamples.shape[:-1], 2, -1)
print(subsamples.shape)
print(subsamples[4, 19])

In [ ]:
ci_strs = np.array([bitstring_matrix_to_integers(samples[:, i]) for samples in subsamples for i in range(2)])
ci_strs = ci_strs.reshape(-1, 2, ci_strs.shape[-1])
print(ci_strs.shape)
print(ci_strs[0])

In [ ]:
def build_subspace(subsamples, include=np.empty((2, 0), dtype=int), max_dim=(None, None)):

    subspaces = []
    for samples in subsamples:
        subspace_list = []
        for i in range(2):

            # 単一スピンのビット列とカウントを取得する。
            bitstrings, counts = np.unique(samples[i], return_counts=True)

            # 単一スピンのビット列を周辺確率の降順にソートする。
            bitstrings = bitstrings[np.argsort(counts)[::-1]]
            
            # 明示的に要求されたビット列を優先し、その後にサンプルされたビット列を続ける
            subspace = np.concatenate((include[i], bitstrings))

            # 元の順序を保ちながら一意な値を取得する
            _, indices = np.unique(subspace, return_index=True)
            indices.sort()
            subspace = subspace[indices]

            # ビット列を最大次元に切り詰める。
            subspace = subspace[:max_dim[i]]
            subspace.sort()

            subspace_list.append(subspace)
        subspaces.append(subspace_list)

    return subspaces

subspace = build_subspace(ci_strs, max_dim=(15, 15))
print(np.array(subspace).shape)

### SQD ループのパラメーター

自己無撞着（self-consistent）ループを実行する前に、各パラメーターが何を制御するかを示します。

| パラメーター | 意味 |
|---|---|
| `max_iterations` | 回復 → サブサンプル → 対角化 のサイクルを何回実行するか。反復が多いほど占有数の推定が収束します。 |
| `num_batches` | 反復ごとの独立した部分空間バッチの数。バッチが多いほど → よりよい統計（箱ひげ図の範囲が広がる）。 |
| `samples_per_batch` | バッチあたりの配置数。大きいほど → 豊かな部分空間だが、古典的な固有値ソルバーの仕事が増える。 |
| `max_dim` | 部分空間内のスピンセクターあたりのスレーター行列式の最大数。ハミルトニアン行列の次元を制御する。 |

In [ ]:
# シード
seed=142

# SQD のオプション
max_iterations = 5

# 固有状態ソルバーのオプション
num_batches = 5
samples_per_batch = 500
max_dim = (300, 300)
max_cycle = 200

# 組み込みの固有値ソルバーにオプションを渡す。デフォルトをそのまま使いたい場合は、
# このステップを省略できる。その場合、下の diagonalize_fermionic_hamiltonian の
# 呼び出しで sci_solver 引数を指定しないことになる。
sci_solver = partial(solve_sci_batch, spin_sq=0.0, max_cycle=max_cycle)

# 途中結果を記録するためのリスト
result_history = []

def callback(results: list[SCIResult]):
    result_history.append(results)
    iteration = len(result_history)
    print(f"Iteration {iteration}")
    for i, result in enumerate(results):
        print(f"\tSubsample {i+1}")
        print(f"\t\tEnergy: {result.energy + nuclear_repulsion_energy}")
        print(f"\t\tSubspace dimension: {np.prod(result.sci_state.amplitudes.shape)}")

### 自己無撞着な SQD ループ

下のループは、4ステップの SQD サイクルを端から端まで実装します。

1. **回復** — *現在の* `current_occupancies` を反転確率の事前分布として `recover_configurations` を呼ぶ。
2. **サブサンプル** — `samples_per_batch` 個の配置の独立したバッチを `num_batches` 個引く。
3. **部分空間を構築** — CI 文字列に変換し、`build_subspace` を呼んでスレーター行列式の基底を得る。
4. **対角化** — `sci_solver` を呼び、その部分空間の中で $\hat{H}$ を射影して対角化する。

対角化の後、鍵となるフィードバックのステップは次のとおりです。
```python
current_occupancies = current_result.orbital_occupancies
```
ソルバーが生成した固有ベクトルは、近似基底状態において各軌道が平均してどれだけ占有されているかを教えてくれます。
この更新された占有数の事前分布が、*次の* 反復のステップ1にフィードバックされ、配置回復を次第に正確にしていきます。
ループは `max_iterations` 回のサイクルを実行し、すべてのバッチと反復にわたって見られた最良（最低エネルギー）の結果を追跡します。

In [ ]:
result_history = []

rng = np.random.default_rng(seed)
current_occupancies = initial_occupancy
best_result = None
sci_solver = solve_sci_batch

# BitArray をビット列と確率の配列に変換する
raw_bitstrings, raw_probs = bit_array_to_arrays(bit_array)

# 配置回復のループを実行する
for _ in range(max_iterations):
    reshaped_bitstrings = reshape_bitstring(raw_bitstrings, num_orbitals)

    # 平均軌道占有数の情報がある場合は、それを使って
    # ノイズを含む配置の全体を洗練する
    bitstrings, probs = recover_configurations(
        reshaped_bitstrings, raw_probs, current_occupancies, num_orbitals, nelec, rand_seed=rng
    )

    # ビット列のバッチをサブサンプルする
    subsamples = np.array(subsample(
        bitstrings.reshape(-1, 2*num_orbitals),
        probs,
        samples_per_batch=samples_per_batch,
        num_batches=num_batches,
        rand_seed=rng,
    ))
    subsamples = subsamples.reshape(*subsamples.shape[:-1], 2, -1)

    ci_strs = np.array([bitstring_matrix_to_integers(samples[:, i]) for samples in subsamples for i in range(2)])
    ci_strs = ci_strs.reshape(-1, 2, ci_strs.shape[-1])

    # ビット列を CI 文字列に変換し、要求された文字列と引き継ぎの文字列を含める
    subspace = build_subspace(ci_strs, max_dim=max_dim)

    # 対角化を実行する
    results = sci_solver(subspace, hcore, eri, num_orbitals, nelec)

    # コールバック関数を呼ぶ
    callback(results)

    # バッチから最良の結果を取得する
    best_result_in_batch = min(results, key=lambda result: result.energy)

    # これまでに見た中で最も低いエネルギーかどうかを確認する
    if best_result is None or best_result_in_batch.energy < best_result.energy:
        best_result = copy.deepcopy(best_result_in_batch)

    # 現在の結果と占有数を保存する
    current_result = copy.deepcopy(best_result_in_batch)
    current_occupancies = current_result.orbital_occupancies

# 少なくとも1回は反復があったはずなので、best_result は None ではない
best_result = cast(SCIResult, best_result)

In [ ]:
print(f"Best energy = {best_result.energy + nuclear_repulsion_energy}")

plt.axhline(scf.e_tot, color="C0", label="HF")
plt.axhline(ccsd.e_tot, color="C1", label="CCSD")
plt.boxplot(
    [x.energy + nuclear_repulsion_energy for results in result_history for x in results],
    tick_labels=['Original SQD'],
)
plt.legend(loc='upper right')
plt.show()

# 最良の部分空間の基底を投稿する
best_subspace = (best_result.sci_state.ci_strs_a, best_result.sci_state.ci_strs_b)

元の SQD 法が HF 状態よりも良いエネルギー推定値を見つけることが分かるかもしれません。これはすでに自明でない結果です。量子サンプルが対角化をヒルベルト空間の物理的に意味のある領域へ導いていることを示すからです。
しかし、この結果はまだ古典的な CCSD のベンチマークほど良くはありません。

SQD と CCSD の差には多くの原因がありえます。最も直接的なてこは **部分空間のサイズ** です。より大きな部分空間は基底状態の波動関数のより多くを捉えますが、それを対角化するにはかなり多くの古典計算が必要になります。スケールアップには、高性能ワークステーションから本格的な HPC クラスターまで、さまざまなものが必要になりえます。

ここでは代わりに、この差のもう1つの主要な要因、すなわち純粋なサンプリングの限界に焦点を当てます。ハードウェアノイズと有限のショット数のため、サンプラーは化学的に重要な電子配置の一部を見逃します。特に、HF 参照から軌道占有数が1つか2つしか違わない配置です。これらの HF に近い配置は基底状態に強く寄与しますが、ハードウェアノイズによって測定分布の中で抑制されることがあります。

**演習4はこの差に直接対処します**。古典的に列挙された参照行列式の集合でサンプルされた部分空間を補強することによってです。

## 演習4: SQD を古典的な参照部分空間で補強する

### 参照部分空間

古典的な部分空間と量子的な部分空間を組み合わせる前に、まず、*純粋に古典的で物理的に動機づけられた* 部分空間の中で対角化するとどうなるかを見てみましょう。量子サンプルはまったく使いません。

背景で紹介したように、**古典的な selected-CI** は、化学者が摂動論や物理的直観を使って、最も影響力のあるスレーター行列式を反復的に特定して取り込む手法の一群です。結果の精度は、その選択の品質に直接依存します。

**このラボの参照部分空間** は、この手法のより単純な総当たりの代用品です。何らかの選択基準を適用するのではなく、最も低いエネルギーから軌道を埋めていくことで低励起ランクの配置をすべて列挙し、最初の 300 個を保ちます。これらの **参照行列式** は構築するのに化学的な洞察を必要とせず、単に系統的な列挙だけです。これにより、再現可能で計算的に安価な古典的ベースラインになります。

背景で述べたように、この総当たりのアプローチは典型的には HF より良いが CCSD には及ばないエネルギーを達成します。これは私たちの **量子有用性のための古典的ベースライン** として機能します。あなたの SQD の結果がこれを上回れば、量子サンプルが、古典的な列挙だけでは見逃す配置を寄与したことになります。

`itertools.combinations` を使って、これらの行列式を系統的に列挙します。

In [ ]:
from itertools import combinations

print(list(combinations(range(5), 3)))

<a id="exercise_4"></a>
<div class="alert alert-block alert-success">

<b>演習4: 古典的な参照部分空間を構築する</b>

**目標:** `ref_bitstrings`（HF 参照の周りの最も低い励起ランクの配置を表す、最大 300 個のスレーター行列式）を構築し、それらを固有値ソルバーに適した `ref_ci_strings` に変換してください。

**背景:** コンサートホールのたとえに戻ると、量子サンプラーが最も重要な着席配置に偶然行き当たるのを期待する代わりに、すべての *最前列* の配置（電子が最も低い利用可能な軌道から先に埋まるもの）を系統的に列挙し、それらを直接固有値ソルバーに渡します。化学者が摂動論を使って最も影響力のある行列式を特定する真の古典的 selected-CI とは異なり、これは総当たりの列挙です。選択基準は適用されず、最初の 300 個の最も低い励起ランクの配置だけです。背景で述べたように、これは典型的には HF と CCSD の間のエネルギーを与え、打ち負かすべき古典的ベースラインとして機能します。

**ヒント:**
- 完全に占有されたビット列から始め、最もエネルギーの高い軌道から先に非占有にしていくことを検討してください。
- `bitstring_matrix_to_integers` を適用して CI 文字列形式に変換してください。
- 対角化器のために α と β の両方の文字列を提供することを忘れないでください。

</div>

In [ ]:
# ---- TODO : タスク4 ----



# ---- TODO : タスク4 ここまで ----

In [ ]:
grade_lab4c_ex4(ref_ci_strings)

では、この純粋に古典的な参照部分空間の中でハミルトニアンを対角化してみましょう。

In [ ]:
ref_result = sci_solver([ref_ci_strings], hcore, eri, num_orbitals, nelec)[0]
print(f"Reference Subspace Energy: {ref_result.energy + nuclear_repulsion_energy}")
print(f"Subspace dimension: {np.prod(ref_result.sci_state.amplitudes.shape)}")

参照部分空間のエネルギーが、元の（純粋な量子サンプリングの）SQD の結果よりも **低い** ことが分かるかもしれません。
これは、手で選んだ低エネルギーの配置が基底状態にとって本当に重要であること、そして量子サンプラーだけではハードウェアノイズのためにそれらの一部を取りこぼす可能性があることを裏付けます。

### 量子サンプルと古典的な参照部分空間を組み合わせる

最も強力なアプローチは、2つの源を組み合わせます。
- 参照行列式の集合を保証されたアンカーとして **固定** する（`include = ref_ci_strings[:, :100]` — 最初の 100 個の配置）。
- 量子サンプラーには、いつものようにバッチあたり残りの `max_dim - 100` 個の配置を供給させる。

すると、各対角化は、量子デバイスが空間の補完的な部分を探索する間、堅固な古典的基盤から始まります。
結果は、純粋サンプリングの場合と純粋参照の場合と比べてどうなると予想しますか？

In [ ]:
# 途中結果を記録するためのリスト
pm_result_history = []

def callback(results: list[SCIResult]):
    pm_result_history.append(results)
    iteration = len(pm_result_history)
    print(f"Iteration {iteration}")
    for i, result in enumerate(results):
        print(f"\tSubsample {i}")
        print(f"\t\tEnergy: {result.energy + nuclear_repulsion_energy}")
        print(f"\t\tSubspace dimension: {np.prod(result.sci_state.amplitudes.shape)}")

# SQD ループの間、最も低い励起ランクの参照行列式 100 個を固定する。
# 残りのエントリーは、いつものように量子サンプルから埋める。
include = ref_ci_strings[:, :100]  # 参照行列式 100 個を固定する

rng = np.random.default_rng(seed)
current_occupancies = initial_occupancy
best_result = None
sci_solver = solve_sci_batch

# BitArray をビット列と確率の配列に変換する
raw_bitstrings, raw_probs = bit_array_to_arrays(bit_array)

# 配置回復のループを実行する
for _ in range(max_iterations):
    reshaped_bitstrings = reshape_bitstring(raw_bitstrings, num_orbitals)

    # 平均軌道占有数の情報がある場合は、それを使って
    # ノイズを含む配置の全体を洗練する
    bitstrings, probs = recover_configurations(
        reshaped_bitstrings, raw_probs, current_occupancies, num_orbitals, nelec, rand_seed=rng
    )

    # ビット列のバッチをサブサンプルする
    subsamples = np.array(subsample(
        bitstrings.reshape(-1, 2*num_orbitals),
        probs,
        samples_per_batch=samples_per_batch,
        num_batches=num_batches,
        rand_seed=rng,
    ))
    subsamples = subsamples.reshape(*subsamples.shape[:-1], 2, -1)

    ci_strs = np.array([bitstring_matrix_to_integers(samples[:, i]) for samples in subsamples for i in range(2)])
    ci_strs = ci_strs.reshape(-1, 2, ci_strs.shape[-1])

    # ビット列を CI 文字列に変換し、要求された文字列と引き継ぎの文字列を含める
    subspace = build_subspace(ci_strs, include, max_dim=max_dim)

    # 対角化を実行する
    results = sci_solver(subspace, hcore, eri, num_orbitals, nelec)

    # コールバック関数を呼ぶ
    callback(results)

    # バッチから最良の結果を取得する
    best_result_in_batch = min(results, key=lambda result: result.energy)

    # これまでに見た中で最も低いエネルギーかどうかを確認する
    if best_result is None or best_result_in_batch.energy < best_result.energy:
        best_result = copy.deepcopy(best_result_in_batch)

    # 現在の結果と占有数を保存する
    current_result = copy.deepcopy(best_result_in_batch)
    current_occupancies = current_result.orbital_occupancies

# 少なくとも1回は反復があったはずなので、best_result は None ではない
best_result = cast(SCIResult, best_result)

In [ ]:
print(f"Best energy (reference-augmented SQD) = {best_result.energy + nuclear_repulsion_energy}")

plt.axhline(scf.e_tot, color="C0", label="HF")
plt.axhline(ccsd.e_tot, color="C1", label="CCSD")
plt.axhline(ref_result.energy + nuclear_repulsion_energy, color="C2", label="Reference Subspace")
plt.boxplot(
    [[x.energy + nuclear_repulsion_energy for results in result_history for x in results],
     [x.energy + nuclear_repulsion_energy for results in pm_result_history for x in results]],
    tick_labels=["Original SQD", "Reference-Augmented SQD"],
)
plt.legend(loc='upper right')
plt.show()

**参照補強 SQD（Reference-Augmented SQD）** の箱ひげ図で、2つの改善に注目してください。

1. **より低いエネルギー** — 最良の推定値が CCSD 参照により近くなります。固定された 100 個の参照行列式が、サンプリングノイズにかかわらず、最も重要な HF に近い配置が常に部分空間にあることを保証するからです。
2. **より狭い広がり** — 箱ひげ図が狭くなり、結果が異なるバッチにわたってより *安定* であることを意味します。参照行列式で部分空間を固定することで、運の悪いサンプリングによる分散が減ります。

**参照補強 SQD のエネルギーが、純粋な古典的参照部分空間のエネルギーより低ければ、おめでとうございます。あなたは実機の量子ハードウェア上で量子有用性を実証したのです。** 量子サンプルが、どんな総当たりの列挙も単独では見つけられない配置を寄与し、それらの配置が測定可能なほど良い基底状態エネルギーの推定値を生みました。これがまさに、実際の量子有用性の姿です。量子デバイスが、古典的な列挙だけでは再現できない何かを加えたのです。よくできました。

## ボーナスチャレンジ: エネルギー推定をさらに改善する

### バッジには必須ではありません

さあ、あなたの番です。エネルギー推定を **量子優位性** に向けてさらに押し進めましょう！

<a id="exercise_bonus"></a>
<div class="alert alert-block alert-success">

<b>ボーナス演習: 自分の最良の部分空間を見つけよう！</b>

**目標:** N₂ の SQD エネルギー推定値をできるだけ低く押し下げてください。

**制約:** 部分空間の次元は **90,000** を超えてはいけません（すなわち `max_dim` を高々 `(300, 300)` に保つ）。この制限は、公正な比較を保証するためにすべての参加者に適用されます。

**ヒント:**

- **戦略的な部分空間の洗練:** 現在の SQD ループが反復にわたって結果を改善していないことに気づいたかもしれません。より戦略的に部分空間を洗練することを検討してください。おそらく、いくつかの質の高い行列式を保ち、SQD の反復をまたいで引き継ぐとよいでしょう。

- **異なる配置回復の方式:** 配置を回復するために使う関数が、基本的にサンプル空間の品質を左右します。`prob_flip_0_to_1` の異なる関数形を試して、回復される配置をより質の高いサンプルへ導いてください。

- **より良いアンザッツ:** より多くの LUCJ 層や、さらに最適化されたパラメーターを使うと、量子サンプルの品質を改善できます。上の補強法で見たように、サンプルの品質は SQD 全体の性能と収束率に大きく影響します。

- **より多くの反復:** 準備ができたと感じたら、`max_iterations` や `num_batches` を増やして、SQD ループに配置をさらに回復させてください。

**期待されること:** 残念ながら、与えられた部分空間のサイズでは、CCSD の結果を打ち負かすことはできないかもしれません。しかし、より大きな部分空間の次元にスケールアップする前に、SQD アルゴリズムを改善するいくつかの重要な技術を練習することはできます。このボーナスチャレンジでは、0 から 100 の範囲のスコアを参照してください。100 が最良の結果です。あなたのスコアは、エネルギーをどれだけうまく最小化するかを反映します。できるならスコア 100 を目指してください！

下のセルで、自分の最良の結果を投稿できます。

</div>

In [ ]:
best_energy = best_result.energy + nuclear_repulsion_energy
best_subspace = (best_result.sci_state.ci_strs_a, best_result.sci_state.ci_strs_b)
grade_lab4c_exbonus(best_energy, best_subspace)  # Submit the best energy and subspace

よくできました。ボーナスチャレンジを完了しました！

# 追加情報

**作成者:** Boseong Kim

**バージョン:** 1.0.0